In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load the dataset
dataset = pd.read_csv(r"C:\Users\krish\Internal-Fraud-Detection-System\datasets\insider_threat_clean_dataset.csv")
print(f"Dataset Shape: {dataset.shape}")

# Create working DataFrame copy
df = dataset.copy()
df.head()

Dataset Shape: (118614, 22)


,employee_department,employee_campus,employee_position,employee_seniority_years,is_contractor,employee_classification,has_foreign_citizenship,has_criminal_record,has_medical_history,employee_origin_country,...,total_files_burned,burned_from_other,is_abroad,trip_day_number,hostility_country_level,num_entries,num_unique_campus,late_exit_flag,entry_during_weekend,is_malicious
0,Engineering Department,Campus C,Design Engineer,22,0,2,0,0,0,Georgia,...,4,0,0,0.0,0,1,1,0,1,0
1,Engineering Department,Campus C,Design Engineer,22,0,2,0,0,0,Georgia,...,0,0,0,0.0,0,1,1,0,0,0
2,Engineering Department,Campus C,Design Engineer,22,0,2,0,0,0,Georgia,...,2,0,0,0.0,0,1,1,0,0,0
3,Engineering Department,Campus C,Design Engineer,22,0,2,0,0,0,Georgia,...,0,0,0,0.0,0,1,1,0,0,0
4,Engineering Department,Campus C,Design Engineer,22,0,2,0,0,0,Georgia,...,0,0,0,0.0,0,0,0,0,0,0


In [3]:
# 1. Create a copy so we don't modify the original data
#df = dataset.copy()

# 2. Check for missing values
#print("Missing values per column:")
#print(df.isnull().sum())

# 3. Automatically find all text (categorical) columns and encode them
#le = LabelEncoder()
#encoded_cols = []

#for col in df.columns:
    #if df[col].dtype == 'object':  # If the column contains text/strings
        #df[col] = le.fit_transform(df[col].astype(str))
        #encoded_cols.append(col)

#print(f"\nSuccessfully encoded text columns: {encoded_cols}")
#print("Data transformed successfully! Ready for splitting.")
#df.head()

In [1]:
#updated encoding cell 
#import joblib

#df = dataset.copy()

#insider_encoders = {}   # will hold {column_name: fitted LabelEncoder}

#for col in df.columns:
    #if df[col].dtype == 'object' and col != 'is_malicious':
        #encoder = LabelEncoder()                     # NEW encoder every iteration
        #df[col] = encoder.fit_transform(df[col].astype(str))
        #insider_encoders[col] = encoder               # keep it

#joblib.dump(insider_encoders, "../models/insider_encoders.pkl")
#print("Encoded + saved columns:", list(insider_encoders.keys()))

KeyboardInterrupt: 

In [13]:
import joblib
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "employee_department",
    "employee_position",
    "employee_campus",
    "employee_origin_country",
]

insider_encoders = {}

# Fit and transform each categorical feature
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    insider_encoders[col] = le

# Save the encoders to the parent models folder (../models/)
joblib.dump(insider_encoders, "../models/insider_encoders.pkl")
print("✅ Saved insider_encoders.pkl successfully to ../models/")

✅ Saved insider_encoders.pkl successfully to ../models/


In [14]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Separate features (X) and target variable (y)
X = df.drop("is_malicious", axis=1)
y = df["is_malicious"]

# 2. Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train Random Forest
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
print("Insider Model trained successfully!\n")

# 4. Evaluate
predictions = model.predict(X_test)
print("--- Classification Report ---")
print(classification_report(y_test, predictions))

# 5. Save Model
with open("../models/insider_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ Saved insider_model.pkl successfully to ../models/")

Insider Model trained successfully!

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     22463
           1       0.75      0.67      0.71      1260

    accuracy                           0.97     23723
   macro avg       0.87      0.83      0.85     23723
weighted avg       0.97      0.97      0.97     23723

✅ Saved insider_model.pkl successfully to ../models/


In [15]:
#check cell
import os
import time

# List of files we want to check inside the models folder
files_to_check = [
    os.path.join("..", "models", "insider_encoders.pkl"),
    os.path.join("..", "models", "insider_model.pkl"),
]

print("--- 🔍 Checking Model Artifact Timestamps ---")
for filepath in files_to_check:
    if os.path.exists(filepath):
        mtime = os.path.getmtime(filepath)
        print(f"✅ Found: {os.path.abspath(filepath)}")
        print(f"   Last Modified: {time.ctime(mtime)}\n")
    else:
        print(f"❌ Not Found: {filepath}\n")

--- 🔍 Checking Model Artifact Timestamps ---
✅ Found: c:\Users\krish\Internal-Fraud-Detection-System\models\insider_encoders.pkl
   Last Modified: Sun Aug  2 15:03:46 2026

✅ Found: c:\Users\krish\Internal-Fraud-Detection-System\models\insider_model.pkl
   Last Modified: Sun Aug  2 15:04:16 2026



In [16]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Assuming X_test and y_test were used for evaluating your model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Print detailed evaluation report
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- ROC-AUC Score ---")
print(roc_auc_score(y_test, y_proba))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     22463
           1       0.75      0.67      0.71      1260

    accuracy                           0.97     23723
   macro avg       0.87      0.83      0.85     23723
weighted avg       0.97      0.97      0.97     23723

--- ROC-AUC Score ---
0.9810017743463855
--- Confusion Matrix ---
[[22184   279]
 [  415   845]]
